# Task 1 - Chest X-ray enhancement
In this notebook I take one dark (underexposed) chest X-ray and try different enhancement techniques on it, one by one.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### Step 1: load the image
X-ray has only one channel so I read it as grayscale.

In [2]:
img = cv2.imread('data/sample_xray.png', 0)
print(img.shape, img.min(), img.max())

plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('original')
plt.axis('off')
plt.show()
cv2.imwrite('output/1_original.png', img)

(512, 512) 2 68


True

### Step 2: histogram equalization
The image is dark so all pixel values are packed in a small range. Equalization spreads them over 0-255 which increases contrast.

In [3]:
eq = cv2.equalizeHist(img)
cv2.imwrite('output/2_equalized.png', eq)

plt.figure(figsize=(12,6))
plt.subplot(2,2,1); plt.imshow(img, cmap='gray', vmin=0, vmax=255); plt.title('original')
plt.subplot(2,2,2); plt.imshow(eq, cmap='gray', vmin=0, vmax=255); plt.title('equalized')
plt.subplot(2,2,3); plt.hist(img.ravel(), 256, [0,256]); plt.title('hist before')
plt.subplot(2,2,4); plt.hist(eq.ravel(), 256, [0,256]); plt.title('hist after')
plt.tight_layout()
plt.savefig('output/2_hist_compare.png')
plt.show()

/tmp/ipykernel_107/2808244029.py:7: MatplotlibDeprecationWarning: Passing the range parameter of hist() positionally is deprecated since Matplotlib 3.10; the parameter will become keyword-only in 3.12.
  plt.subplot(2,2,3); plt.hist(img.ravel(), 256, [0,256]); plt.title('hist before')
/tmp/ipykernel_107/2808244029.py:8: MatplotlibDeprecationWarning: Passing the range parameter of hist() positionally is deprecated since Matplotlib 3.10; the parameter will become keyword-only in 3.12.
  plt.subplot(2,2,4); plt.hist(eq.ravel(), 256, [0,256]); plt.title('hist after')


**My observation:** (write here what you see - e.g. lungs, ribs and heart outline are clearer after equalization and the histogram is now spread out)

### Step 3: color map (JET)
Colors are easier for our eyes to tell apart than shades of gray, so small intensity differences (like fluid edges) become easier to notice.

In [4]:
jet = cv2.applyColorMap(eq, cv2.COLORMAP_JET)
cv2.imwrite('output/3_jet.png', jet)

plt.imshow(cv2.cvtColor(jet, cv2.COLOR_BGR2RGB))
plt.title('JET heatmap')
plt.axis('off')
plt.show()

### Step 4: color balance
The JET image has a color cast. To fix it I scale each channel so that all three have the same average (gray world idea).

In [5]:
b, g, r = cv2.split(jet.astype(np.float32))
avg = (b.mean() + g.mean() + r.mean()) / 3

b = b * avg / b.mean()
g = g * avg / g.mean()
r = r * avg / r.mean()

balanced = np.clip(cv2.merge([b, g, r]), 0, 255).astype(np.uint8)
cv2.imwrite('output/4_balanced.png', balanced)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(jet, cv2.COLOR_BGR2RGB)); plt.title('before')
plt.subplot(1,2,2); plt.imshow(cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)); plt.title('after')
plt.show()

### Step 5: thresholding
Bones and dense fluid are the brightest parts, so I keep only the top 15% brightest pixels and make the rest black.

In [6]:
t = np.percentile(eq, 85)
print('threshold value:', t)

_, mask = cv2.threshold(eq, t, 255, cv2.THRESH_BINARY)
dense = cv2.bitwise_and(eq, eq, mask=mask)
cv2.imwrite('output/5_dense.png', dense)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1); plt.imshow(mask, cmap='gray'); plt.title('mask')
plt.subplot(1,2,2); plt.imshow(dense, cmap='gray'); plt.title('dense tissue only')
plt.show()

threshold value: 217.0


### Step 6: log transform
Formula: s = c * log(1 + r). It makes dark pixels brighter a lot, so I applied it on the original dark image to see the faint ribcage edges.

In [7]:
c = 255 / np.log(1 + img.max())
log_img = (c * np.log(1 + img.astype(np.float32))).astype(np.uint8)
cv2.imwrite('output/6_log.png', log_img)

plt.figure(figsize=(10,5))
plt.subplot(1,2,1); plt.imshow(img, cmap='gray', vmin=0, vmax=255); plt.title('original')
plt.subplot(1,2,2); plt.imshow(log_img, cmap='gray', vmin=0, vmax=255); plt.title('log')
plt.show()

### Step 7: gamma (power law)
Formula: s = 255 * (r/255)^gamma. With gamma < 1 the midtones get brighter and the bone contrast becomes softer. I tried 3 values.

In [8]:
def gamma_fix(image, gamma):
    table = np.array([255 * (i/255)**gamma for i in range(256)]).astype(np.uint8)
    return cv2.LUT(image, table)

plt.figure(figsize=(14,4))
plt.subplot(1,4,1); plt.imshow(img, cmap='gray', vmin=0, vmax=255); plt.title('original')
for i, gm in enumerate([0.3, 0.5, 0.7]):
    out = gamma_fix(img, gm)
    cv2.imwrite(f'output/7_gamma_{gm}.png', out)
    plt.subplot(1,4,i+2); plt.imshow(out, cmap='gray', vmin=0, vmax=255); plt.title(f'gamma {gm}')
plt.show()

### Conclusion
(write 2-3 lines in your own words: which step helped most and why - e.g. gamma 0.5 looked best for soft lung tissue)